In [5]:
"""
Generate simulated ADF-STEM images for bilayer ReS2 and the corresponding
single-layer components.

This script generates:
1. layer1 monolayer ReS2 image;
2. layer2 monolayer ReS2 image after in-plane translation;
3. bilayer ReS2 image formed by stacking layer1 and layer2.

Output database structure
-------------------------
DATABASE_PATH/
├── layer1/
│   └── data/
├── layer2/
│   └── data/
└── bilayer/
    └── data/

All paths are controlled only in the main execution section.
"""

from ase.io import read, write
from ase.build import molecule, rotate, cut
from ase.visualize import view
from ase import Atoms
import torchvision.transforms.functional as TF
from collections import Counter
import subprocess
import os
import cv2
import random
import numpy as np
from PIL import Image

os.environ["KMP_DUPLICATE_LIB_OK"] = "True"


def to_database(database_path="", dat_path=""):
    """
    Convert simulated tif images recorded in a dat file into cropped png images.

    Parameters
    ----------
    database_path : str
        Output database path. Images are saved into:
            database_path/data/

    dat_path : str
        Text file containing simulation folder paths.
        Each line should point to a folder containing an image/ subfolder.

    Current behavior
    ----------------
    - Only image data are saved.
    - Label saving is disabled.
    - Each image is cropped into a 1024 x 1024 patch before saving.
    """
    data_output_dir = os.path.join(database_path, "data")
    os.makedirs(data_output_dir, exist_ok=True)

    with open(dat_path, "r") as data_paths:
        data_path = data_paths.readlines()

        for i in range(len(data_path)):
            current_path = data_path[i].strip()
            image_dir = os.path.join(current_path, "image")

            if not os.path.exists(image_dir):
                print(f"[Warning] Image directory not found: {image_dir}")
                continue

            for img in os.listdir(image_dir):
                image_path = os.path.join(image_dir, img)

                if not os.path.isfile(image_path):
                    continue

                if not img.lower().endswith((".tif", ".tiff")):
                    continue

                image = Image.open(image_path)

                # Keep original crop behavior.
                image = TF.crop(
                    image,
                    image.size[0] // 2 - 800,
                    image.size[1] // 2 - 300,
                    1024,
                    1024,
                )

                output_name = os.path.splitext(img)[0] + ".png"
                image.save(os.path.join(data_output_dir, output_name), "PNG")


def write_xyz(structure, file_name="", no_show_s=True):
    """
    Write an ASE Atoms object into the custom XYZ-like format required by incoSTEM.

    Parameters
    ----------
    structure : ase.Atoms
        Atomic structure to export.

    file_name : str
        Output xyz file path.

    no_show_s : bool
        If True, S atoms are omitted from the exported incoSTEM input.
    """
    file = ""

    symbols = structure.get_chemical_symbols()
    atom_counts = Counter(symbols)

    for atom, count in atom_counts.items():
        file += atom + str(count)

    file += "\t" + str(len(structure)) + "\n"

    cell = structure.get_cell()
    file += (
        str(cell[0][0])
        + "\t"
        + str(cell[1][1])
        + "\t"
        + str(cell[2][2])
        + "\n"
    )

    write("temp.xyz", structure)

    with open("temp.xyz", "r") as p:
        content = p.read().splitlines()

        i = 2
        for line in content:
            if i >= 0:
                i -= 1
                continue

            if ("S" in line) and no_show_s:
                continue

            line = line.replace("S", "16")
            line = line.replace("Re", "75")

            a = [value for value in line.split(" ") if value is not None and value != ""]

            for item in a:
                file += item + "\t"

            file += "1" + "\t"
            file += "0" + "\t" + "\n"

        file += "-1"

    with open(file_name, "w") as f:
        f.write(file)


def save_label(size, structure, file_name, p_size=10):
    """
    Save a binary Re atom label image for a given structure.

    This function is kept from the original code but is not used in the current
    execution workflow.
    """
    img = np.zeros(
        [int(structure.cell[1][1] / (structure.cell[0][0] / size)), size],
        np.uint8,
    )

    for atom in structure:
        if atom.symbol == "Re":
            p = atom.position[:2] / (structure.cell[0][0] / size)
            p = p.astype(int)
            cv2.circle(img, p, p_size, (255, 255, 255), -1)

    cv2.imwrite(file_name + ".png", img)


def save_train_label(size, layer1, layer2, file_name, p_size=15):
    """
    Save a three-channel layer-resolved Re label image for bilayer ReS2.

    Channel 0: Re atoms in layer1.
    Channel 1: Re atoms in layer2.
    Channel 2: empty channel.

    This function is kept from the original code, but the call is currently
    commented out in generate_data().
    """
    if layer1.cell[1][1] > layer1.cell[0][0]:
        img = np.zeros(
            [int(layer1.cell[1][1] / (layer1.cell[0][0] / size)), size],
            np.uint8,
        )
    else:
        img = np.zeros(
            [size, int(layer1.cell[0][0] / (layer1.cell[1][1] / size))],
            np.uint8,
        )

    img1 = img.copy()
    img2 = img.copy()

    for atom in layer1:
        if atom.symbol == "Re":
            if layer1.cell[1][1] > layer1.cell[0][0]:
                p = atom.position[:2] / (layer1.cell[0][0] / size)
            else:
                p = atom.position[:2] / (layer1.cell[1][1] / size)

            p = p.astype(int) + 1
            cv2.circle(img1, p, p_size, (255, 255, 255), -1)

    for atom in layer2:
        if atom.symbol == "Re":
            if layer1.cell[1][1] > layer1.cell[0][0]:
                p = atom.position[:2] / (layer2.cell[0][0] / size)
            else:
                p = atom.position[:2] / (layer2.cell[1][1] / size)

            p = p.astype(int) + 1
            cv2.circle(img2, p, p_size, (255, 255, 255), -1)

    merged_array = np.concatenate(([img1], [img2], [img]), axis=0)
    label = np.transpose(merged_array, (1, 2, 0))

    cv2.imwrite(file_name + ".png", label)


def run_computem(incostem_path, param_path):
    """
    Run incoSTEM simulations for all parameter files in a folder.
    """
    for param in os.listdir(param_path):
        param_file = os.path.join(param_path, param)

        if not os.path.isfile(param_file):
            continue

        process = subprocess.Popen(
            incostem_path,
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            universal_newlines=True,
        )

        with open(param_file, "r") as p:
            lines = p.readlines()
            for line in lines:
                process.stdin.write(line)
                process.stdin.flush()

        output = process.communicate()[0]


def generate_param(
    path="",
    image_size=4096,
    params=["", "", "", ""],
    d_num=5,
    s_num=5,
    layer_name="",
):
    """
    Generate incoSTEM parameter files.

    If layer_name is empty:
        use ReS2.xyz and generate bilayer image.

    If layer_name is 'layer1' or 'layer2':
        use ReS2_layer1.xyz or ReS2_layer2.xyz and generate monolayer image.
    """
    param_dir = os.path.join(path, "param")
    image_dir = os.path.join(path, "image")

    os.makedirs(param_dir, exist_ok=True)
    os.makedirs(image_dir, exist_ok=True)

    A1_param_mean = 1 * 10**-6
    A1_param_std = 1 * 10**-6
    B2_param_mean = 10 * 10**-6
    B2_param_std = 20 * 10**-6
    A2_param_mean = 25 * 10**-6
    A2_param_std = 50 * 10**-6

    C12a = 0
    C12b = 0
    C21a = 0
    C21b = 0
    C23a = 0
    C23b = 0

    pd = 10.0 / d_num
    ps = 0.1 / s_num

    for i in range(d_num):
        for j in range(s_num):
            Defocus = 35 + i * pd
            Source_size_at_specimen = 0.6 + j * ps

            idx = random.uniform(10000, 20000)

            base_name = (
                f"{params[0][:4]}_"
                f"{params[1][:4]}_"
                f"{params[2][:4]}_"
                f"{params[3][:4]}_"
                f"{Defocus}_"
                f"{Source_size_at_specimen}_"
                f"{idx}"
            )

            if layer_name:
                image_filename = f"{base_name}_{layer_name}.tif"
            else:
                image_filename = f"{base_name}_bilayer.tif"

            param_file = os.path.join(param_dir, f"{i}{j}.param")

            with open(param_file, "w") as param:
                xyz_file = "ReS2.xyz" if not layer_name else f"ReS2_{layer_name}.xyz"

                param.write(os.path.join(path, xyz_file) + "\n")
                param.write("1 1 1\n")
                param.write(os.path.join(image_dir, image_filename) + "\n")
                param.write(f"{image_size} {image_size}\n")
                param.write(f"300 0 0 {Defocus} 21.3\n")
                param.write("39 200\n")
                param.write(
                    f"C12a {C12a} C12b {C12b} "
                    f"C21a {C21a} C21b {C21b} "
                    f"C23a {C23a} C23b {C23b} END\n"
                )
                param.write(f"{Source_size_at_specimen}\n")
                param.write("0 \n")
                param.write("n\n")
                param.write("-1\n")


def generate_data(
    structure_path="",
    DAT_path="",
    move_list=None,
    incostem_path="",
    image_size=4096,
    image_num=1,
    no_show_s=True,
    output_data_root="",
    database_path="",
):
    """
    Generate bilayer ReS2 and corresponding single-layer STEM images.

    Generated data
    --------------
    For each move_x/move_y configuration, this function generates:
    1. layer1 single-layer image;
    2. layer2 single-layer image;
    3. bilayer image.

    Database output
    ---------------
    Images are saved separately into:
        database_path/layer1/data/
        database_path/layer2/data/
        database_path/bilayer/data/
    """
    if move_list is None:
        move_list = [0, 0]

    if output_data_root == "":
        raise ValueError("output_data_root must be provided.")

    if database_path == "":
        raise ValueError("database_path must be provided.")

    atoms = read(structure_path)

    ReS2_layer1 = atoms.copy()
    ReS2_layer2 = atoms.copy()

    move_x = move_list[0]
    move_y = move_list[1]
    move_z = 0.67

    ReS2_layer2.positions[:, 0] += move_x
    ReS2_layer2.positions[:, 1] += move_y
    ReS2_layer2.positions[:, 2] += move_z

    # Random rotation is effectively disabled because random.uniform(0, 1) > 2.0 is never True.
    if random.uniform(0, 1) > 2.0:
        r = random.uniform(0.0, 360.0)
    else:
        r = 0

    center = atoms.get_cell().sum(axis=0) / 2.0
    ReS2_layer2.rotate(r, "z", center)

    # Random flip is effectively disabled for the same reason.
    if random.uniform(0, 1) > 2.0:
        if_flip = "1"
        ReS2_layer2.rotate(180, "y", center)
    else:
        if_flip = "0"

    # Optional random defect block is currently disabled.
    # for i in range(image_num):
    #     if len(ReS2_layer1) > 10:
    #         ReS2_layer1.pop(random.randint(0, len(ReS2_layer1) - 10))
    #     if len(ReS2_layer2) > 10:
    #         ReS2_layer2.pop(random.randint(0, len(ReS2_layer2) - 10))

    ReS2 = ReS2_layer1 + ReS2_layer2

    ReS2_layer2 = ReS2[len(ReS2_layer1):]
    ReS2_layer1 = ReS2[:len(ReS2_layer1)]

    path = os.path.join(
        output_data_root,
        f"ReS2_{move_x:.6f}_{move_y:.6f}_{r:.4f}_{if_flip}",
    )
    os.makedirs(path, exist_ok=True)

    os.makedirs(os.path.dirname(DAT_path), exist_ok=True)

    write_xyz(
        ReS2,
        file_name=os.path.join(path, "ReS2.xyz"),
        no_show_s=no_show_s,
    )

    # Label generation is currently disabled.
    # save_train_label(
    #     image_size,
    #     ReS2_layer1,
    #     ReS2_layer2,
    #     file_name=os.path.join(path, "ReS2")
    # )

    # -------------------------------------------------------------------------
    # Generate layer1 and layer2 monolayer images.
    # -------------------------------------------------------------------------
    for layer_name, layer_structure in [
        ("layer1", ReS2_layer1),
        ("layer2", ReS2_layer2),
    ]:
        layer_path = os.path.join(path, layer_name)
        os.makedirs(layer_path, exist_ok=True)

        write_xyz(
            layer_structure,
            file_name=os.path.join(layer_path, f"ReS2_{layer_name}.xyz"),
            no_show_s=no_show_s,
        )

        generate_param(
            layer_path,
            image_size,
            params=[str(move_x), str(move_y), str(r), if_flip],
            d_num=1,
            s_num=1,
            layer_name=layer_name,
        )

        run_computem(incostem_path, os.path.join(layer_path, "param"))

        dat_path_for_layer = os.path.join(layer_path, "ReS2.dat")
        with open(dat_path_for_layer, "w") as dat:
            dat.write(layer_path + "\n")

        # Save layer1 images into database_path/layer1/data/.
        # Save layer2 images into database_path/layer2/data/.
        to_database(
            database_path=os.path.join(database_path, layer_name),
            dat_path=dat_path_for_layer,
        )

    # -------------------------------------------------------------------------
    # Generate bilayer image.
    # -------------------------------------------------------------------------
    generate_param(
        path,
        image_size,
        params=[str(move_x), str(move_y), str(r), if_flip],
        d_num=1,
        s_num=1,
    )

    run_computem(incostem_path, os.path.join(path, "param"))

    # Record bilayer path for final bilayer-only database conversion.
    with open(DAT_path, "a") as dat:
        dat.write(path + "\n")


# ================== Execution section ==================

if __name__ == "__main__":
    # -------------------------------------------------------------------------
    # Only modify paths here.
    # -------------------------------------------------------------------------
    BASE_DIR = r"G:\DiffStack-code"

    STRUCTURE_PATH = os.path.join(
        BASE_DIR,
        "Data_gen",
        "structure",
        "ReS2",
        "one_layer.xyz",
    )

    DAT_PATH = os.path.join(
        BASE_DIR,
        "Symbolic-regression",
        "dat",
        "ReS2.dat",
    )

    OUTPUT_DATA_ROOT = os.path.join(
        BASE_DIR,
        "Symbolic-regression",
        "data",
    )

    DATABASE_PATH = os.path.join(
        BASE_DIR,
        "Symbolic-regression",
        "computem4sym",
    )

    INCOSTEM_PATH = r"G:\Moire_Code\condition_ddpm\generate_data\generate_train_data\incostem.exe"

    # -------------------------------------------------------------------------
    # Other generation settings.
    # -------------------------------------------------------------------------
    IMAGE_SIZE = 2048
    NO_SHOW_S = True

    # Clear previous dat records to avoid mixing old and new paths.
    os.makedirs(os.path.dirname(DAT_PATH), exist_ok=True)
    with open(DAT_PATH, "w") as f:
        f.write("")

    # Make sure output folders exist.
    os.makedirs(OUTPUT_DATA_ROOT, exist_ok=True)
    os.makedirs(DATABASE_PATH, exist_ok=True)

    # Optional: clear old database outputs.
    # Comment these lines if you want to keep old generated png images.
    for subfolder in ["layer1", "layer2", "bilayer"]:
        subfolder_path = os.path.join(DATABASE_PATH, subfolder)
        if os.path.exists(subfolder_path):
            import shutil
            shutil.rmtree(subfolder_path)

    # Scan multiple in-plane translations.
    for x in range(-32, 32, 32):
        for y in range(-28, 28, 28):
            generate_data(
                structure_path=STRUCTURE_PATH,
                DAT_path=DAT_PATH,
                move_list=[x / 10.0, y / 10.0],
                incostem_path=INCOSTEM_PATH,
                image_size=IMAGE_SIZE,
                image_num=1,
                no_show_s=NO_SHOW_S,
                output_data_root=OUTPUT_DATA_ROOT,
                database_path=DATABASE_PATH,
            )

    # Convert all bilayer tif images recorded in the global dat file.
    # Bilayer images are saved into DATABASE_PATH/bilayer/data/.
    to_database(
        database_path=os.path.join(DATABASE_PATH, "bilayer"),
        dat_path=DAT_PATH,
    )

# ReS2 twist

In [6]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from ase.io import read, write
from ase.build import molecule, rotate, cut
from ase.visualize import view
from ase import Atoms
import torchvision.transforms.functional as TF
from collections import Counter
import subprocess

import cv2
import random
import numpy as np
from PIL import Image


# =============================================================================
# 1. Database conversion
# =============================================================================

def to_database(database_path="", dat_path="", layer_suffix=""):
    """
    Convert simulated tif images and their corresponding labels into cropped png files.

    Parameters
    ----------
    database_path : str
        Root database path. Images and labels are saved into:
            database_path / layer_suffix / data
            database_path / layer_suffix / label

    dat_path : str
        A text file containing one or more simulation folder paths.
        Each folder should contain:
            image/
            ReS2_<layer_suffix>.png

    layer_suffix : str
        Layer identifier, such as:
            layer1
            layer2
            bilayer

    Current behavior
    ----------------
    - The simulated image is cropped to 1024 x 1024 before saving.
    - The corresponding label is cropped using the same crop window.
    - Output images and labels are saved into separate layer-specific folders.
    """
    layer_database_path = os.path.join(database_path, layer_suffix)

    data_dir = os.path.join(layer_database_path, "data")
    label_dir = os.path.join(layer_database_path, "label")

    os.makedirs(data_dir, exist_ok=True)
    os.makedirs(label_dir, exist_ok=True)

    with open(dat_path, "r") as data_paths:
        data_path = data_paths.readlines()

        for i in range(len(data_path)):
            current_path = data_path[i].strip()

            label_path = os.path.join(current_path, f"ReS2_{layer_suffix}.png")
            if not os.path.exists(label_path):
                print(f"[Warning] Label file not found: {label_path}")
                continue

            label_n = Image.open(label_path)

            img_dir = os.path.join(current_path, "image")
            if not os.path.exists(img_dir):
                print(f"[Warning] Image directory not found: {img_dir}")
                continue

            for img in os.listdir(img_dir):
                img_path = os.path.join(img_dir, img)

                if not os.path.isfile(img_path):
                    continue

                if not img.lower().endswith((".tif", ".tiff")):
                    continue

                image = Image.open(img_path)

                # Keep the original crop logic: 1024 x 1024 crop.
                image = TF.crop(
                    image,
                    image.size[0] // 2 - 800,
                    image.size[1] // 2 - 300,
                    1024,
                    1024
                )

                label = TF.crop(
                    label_n.copy(),
                    label_n.size[0] // 2 - 800,
                    label_n.size[1] // 2 - 300,
                    1024,
                    1024
                )

                save_img_name = f"{os.path.splitext(img)[0]}_type_{layer_suffix}.png"
                save_label_name = f"{os.path.splitext(img)[0]}_type_{layer_suffix}.png"

                image.save(os.path.join(data_dir, save_img_name), "PNG")
                label.save(os.path.join(label_dir, save_label_name), "PNG")


# =============================================================================
# 2. XYZ export
# =============================================================================

def write_xyz(structure, file_name="", no_show_s=True):
    """
    Write an ASE Atoms object into the custom XYZ-like format required by incoSTEM.

    Parameters
    ----------
    structure : ase.Atoms
        Atomic structure to export.

    file_name : str
        Output xyz file path.

    no_show_s : bool
        If True, S atoms are omitted from the incoSTEM input.

    Element mapping
    ---------------
    Re -> 75
    S  -> 16
    """
    file = ""

    symbols = structure.get_chemical_symbols()
    atom_counts = Counter(symbols)

    for atom, count in atom_counts.items():
        file += atom + str(count)

    file += "\t" + str(len(structure)) + "\n"

    cell = structure.get_cell()
    file += str(cell[0][0]) + "\t" + str(cell[1][1]) + "\t" + str(cell[2][2]) + "\n"

    write("temp.xyz", structure)

    with open("temp.xyz", "r") as p:
        content = p.read().splitlines()

        i = 2
        for line in content:
            if i >= 0:
                i -= 1
                continue

            if ("S" in line) and no_show_s:
                continue

            line = line.replace("S", "16")
            line = line.replace("Re", "75")

            a = [value for value in line.split(" ") if value is not None and value != ""]

            for item in a[:4]:
                file += item + "\t"

            file += "1" + "\t"
            file += "0" + "\t" + "\n"

        file += "-1"

    with open(file_name, "w") as f:
        f.write(file)


# =============================================================================
# 3. Label generation
# =============================================================================

def save_label(size, structure, file_name, p_size=0):
    """
    Save a single-layer Re atom label.

    Parameters
    ----------
    size : int
        Reference image size.

    structure : ase.Atoms
        Single-layer ReS2 structure.

    file_name : str
        Output path without '.png'.

    p_size : int
        Marker radius for Re atoms.
    """
    if structure.cell[1][1] > structure.cell[0][0]:
        img_shape = [
            int(structure.cell[1][1] / (structure.cell[0][0] / size)),
            size
        ]
    else:
        img_shape = [
            size,
            int(structure.cell[0][0] / (structure.cell[1][1] / size))
        ]

    img = np.zeros(img_shape, np.uint8)

    for atom in structure:
        if atom.symbol == "Re":
            if structure.cell[1][1] > structure.cell[0][0]:
                p = atom.position[:2] / (structure.cell[0][0] / size)
            else:
                p = atom.position[:2] / (structure.cell[1][1] / size)

            p = p.astype(int)
            cv2.circle(img, p, p_size, (255, 255, 255), -1)

    cv2.imwrite(file_name + ".png", img)


def save_train_label(size, layer1, layer2, file_name, p_size=10):
    """
    Save a three-channel bilayer ReS2 label.

    Channel design
    --------------
    Channel 0: Re atoms in layer1.
    Channel 1: Re atoms in layer2.
    Channel 2: empty channel.
    """
    if layer1.cell[1][1] > layer1.cell[0][0]:
        img_shape = [
            int(layer1.cell[1][1] / (layer1.cell[0][0] / size)),
            size
        ]
    else:
        img_shape = [
            size,
            int(layer1.cell[0][0] / (layer1.cell[1][1] / size))
        ]

    img = np.zeros(img_shape, np.uint8)

    img1 = img.copy()
    img2 = img.copy()

    for atom in layer1:
        if atom.symbol == "Re":
            if layer1.cell[1][1] > layer1.cell[0][0]:
                p = atom.position[:2] / (layer1.cell[0][0] / size)
            else:
                p = atom.position[:2] / (layer1.cell[1][1] / size)

            p = p.astype(int) + 1
            cv2.circle(img1, p, p_size, (255, 255, 255), -1)

    for atom in layer2:
        if atom.symbol == "Re":
            if layer1.cell[1][1] > layer1.cell[0][0]:
                p = atom.position[:2] / (layer2.cell[0][0] / size)
            else:
                p = atom.position[:2] / (layer2.cell[1][1] / size)

            p = p.astype(int) + 1
            cv2.circle(img2, p, p_size, (255, 255, 255), -1)

    merged_array = np.concatenate(([img1], [img2], [img]), axis=0)
    label = np.transpose(merged_array, (1, 2, 0))

    cv2.imwrite(file_name + ".png", label)


# =============================================================================
# 4. incoSTEM execution
# =============================================================================

def run_computem(incostem_path, param_path):
    """
    Run incoSTEM simulations for all parameter files in a folder.

    Parameters
    ----------
    incostem_path : str
        Path to incostem.exe.

    param_path : str
        Folder containing parameter files.
    """
    for param in os.listdir(param_path):
        param_full_path = os.path.join(param_path, param)

        if not os.path.isfile(param_full_path):
            continue

        process = subprocess.Popen(
            incostem_path,
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            universal_newlines=True
        )

        with open(param_full_path, "r") as p:
            lines = p.readlines()

            for line in lines:
                process.stdin.write(line)
                process.stdin.flush()

        output = process.communicate()[0]


# =============================================================================
# 5. incoSTEM parameter generation
# =============================================================================

def generate_param(path="", image_size=4096, params=["", "", "", ""], d_num=5, s_num=5, layer_type=""):
    """
    Generate incoSTEM parameter files for layer1, layer2 or bilayer ReS2.

    Parameters
    ----------
    path : str
        Folder of the current structure.

    image_size : int
        Simulated image size.

    params : list[str]
        Metadata used in image filename:
            params[0] = move_x
            params[1] = move_y
            params[2] = rotation angle
            params[3] = flip flag

    d_num : int
        Number of defocus values.

    s_num : int
        Number of source-size values.

    layer_type : str
        One of:
            layer1
            layer2
            bilayer
    """
    param_dir = os.path.join(path, "param")
    img_dir = os.path.join(path, "image")

    os.makedirs(param_dir, exist_ok=True)
    os.makedirs(img_dir, exist_ok=True)

    A1_param_mean = 1 * 10 ** -6
    A1_param_std = 1 * 10 ** -6
    B2_param_mean = 10 * 10 ** -6
    B2_param_std = 20 * 10 ** -6
    A2_param_mean = 25 * 10 ** -6
    A2_param_std = 50 * 10 ** -6

    C12a = C12b = C21a = C21b = C23a = C23b = 0

    pd = 10.0 / d_num
    ps = 0.1 / s_num

    for i in range(d_num):
        for j in range(s_num):
            Defocus = 35 + i * pd
            Source_size_at_specimen = 0.6 + j * ps
            idx = random.uniform(10000, 20000)

            param_file_name = f"{i}{j}_{layer_type}.param"
            param_full_path = os.path.join(param_dir, param_file_name)

            xyz_file_name = f"ReS2_type_{layer_type}.xyz"
            xyz_full_path = os.path.join(path, xyz_file_name)

            img_base_name = (
                f"{params[0][:4]}_"
                f"{params[1][:4]}_"
                f"{params[2][:4]}_"
                f"{params[3][:4]}_"
                f"{str(Defocus)[:4]}_"
                f"{str(Source_size_at_specimen)[:4]}_"
                f"{str(idx)[:4]}"
            )

            img_file_name = f"{img_base_name}_type_{layer_type}.tif"
            img_full_path = os.path.join(img_dir, img_file_name)

            with open(param_full_path, "w") as param:
                param.write(xyz_full_path + "\n")
                param.write("1 1 1\n")
                param.write(img_full_path + "\n")
                param.write(f"{image_size} {image_size}\n")
                param.write(f"300 0 0 {Defocus} 21.3\n")
                param.write("39 200\n")
                param.write(
                    f"C12a {C12a} C12b {C12b} "
                    f"C21a {C21a} C21b {C21b} "
                    f"C23a {C23a} C23b {C23b} END\n"
                )
                param.write(f"{Source_size_at_specimen}\n")
                param.write("0 \n")
                param.write("n\n")
                param.write("-1\n")


# =============================================================================
# 6. ReS2 twist-structure generation
# =============================================================================

def generate_data(
    structure_path="",
    DAT_path="",
    move_list=None,
    incostem_path="",
    image_size=4096,
    image_num=1,
    no_show_s=True,
    output_data_root="",
    database_path=""
):
    """
    Generate twist-stacked bilayer ReS2 and its corresponding layer1/layer2 images.

    Parameters
    ----------
    structure_path : str
        Path to the monolayer ReS2 structure file.

    DAT_path : str
        Global DAT file used to record generated main folders.

    move_list : list[float]
        [move_x, move_y, rotation_angle]

    incostem_path : str
        Path to incostem.exe.

    image_size : int
        Simulated image size.

    image_num : int
        Kept for compatibility with the original script.

    no_show_s : bool
        If True, S atoms are omitted from incoSTEM input.

    output_data_root : str
        Root folder for generated raw simulation folders.

    database_path : str
        Root database path. Final data are archived into:
            database_path/layer1/
            database_path/layer2/
            database_path/bilayer/
    """
    if move_list is None:
        move_list = [0, 0, 0]

    if output_data_root == "":
        raise ValueError("output_data_root must be provided.")

    if database_path == "":
        raise ValueError("database_path must be provided.")

    atoms = read(structure_path)

    ReS2_layer1 = atoms.copy()
    ReS2_layer2 = atoms.copy()

    # Layer spacing for bilayer ReS2.
    ReS2_layer2.positions[:, 2] += 6.7

    move_x = move_list[0]
    move_y = move_list[1]
    r = move_list[2]

    # The original random-rotation override is kept.
    # Since random.uniform(0, 1) > 2.0 is never True, r usually remains move_list[2].
    if random.uniform(0, 1) > 2.0:
        r = random.uniform(0.0, 180.0)

    center = atoms.get_cell().sum(axis=0) / 2.0
    ReS2_layer2.rotate(r, "z", center)

    ReS2_layer2.positions[:, 0] += move_x
    ReS2_layer2.positions[:, 1] += move_y

    # The original flip logic is kept.
    # Since random.uniform(0, 1) > 2.0 is never True, flip is normally disabled.
    if_flip = "0"

    if random.uniform(0, 1) > 2.0:
        if_flip = "1"
        ReS2_layer2.rotate(180, "y", center)

    ReS2_bilayer = ReS2_layer1 + ReS2_layer2

    ReS2_layer1 = ReS2_bilayer[:len(atoms)]
    ReS2_layer2 = ReS2_bilayer[len(atoms):]

    main_dir = os.path.join(
        output_data_root,
        f"ReS2_x{str(move_x)[:6]}_y{str(move_y)[:6]}_rot{str(r)[:4]}_flip{if_flip}"
    )

    os.makedirs(main_dir, exist_ok=True)
    os.makedirs(os.path.dirname(DAT_path), exist_ok=True)

    layers_config = [
        ("layer1", ReS2_layer1, "layer1"),
        ("layer2", ReS2_layer2, "layer2"),
        ("bilayer", ReS2_bilayer, "bilayer")
    ]

    for layer_subdir, layer_struct, layer_type in layers_config:
        layer_dir = os.path.join(main_dir, layer_subdir)
        os.makedirs(layer_dir, exist_ok=True)

        xyz_file_name = f"ReS2_type_{layer_type}.xyz"
        xyz_full_path = os.path.join(layer_dir, xyz_file_name)

        write_xyz(
            layer_struct,
            file_name=xyz_full_path,
            no_show_s=no_show_s
        )

        label_file_name = f"ReS2_{layer_type}"

        if layer_type in ["layer1", "layer2"]:
            save_label(
                image_size,
                layer_struct,
                file_name=os.path.join(layer_dir, label_file_name),
                p_size=0
            )
        else:
            save_train_label(
                image_size,
                ReS2_layer1,
                ReS2_layer2,
                file_name=os.path.join(layer_dir, label_file_name),
                p_size=10
            )

        generate_param(
            path=layer_dir,
            image_size=image_size,
            params=[str(move_x), str(move_y), str(r), if_flip],
            d_num=1,
            s_num=1,
            layer_type=layer_type
        )

        run_computem(
            incostem_path,
            os.path.join(layer_dir, "param")
        )

        layer_dat_path = os.path.join(layer_dir, f"ReS2_{layer_type}.dat")

        with open(layer_dat_path, "w") as f:
            f.write(layer_dir + "\n")

        to_database(
            database_path=database_path,
            dat_path=layer_dat_path,
            layer_suffix=layer_type
        )

    with open(DAT_path, "a") as f:
        f.write(main_dir + "\n")


# =============================================================================
# 7. Execution section
# =============================================================================

if __name__ == "__main__":
    # -------------------------------------------------------------------------
    # Only modify paths here.
    # -------------------------------------------------------------------------
    BASE_DIR = r"G:\DiffStack-code"

    STRUCTURE_PATH = os.path.join(
        BASE_DIR,
        "Data_gen",
        "structure",
        "ReS2",
        "one_layer.xyz"
    )

    DAT_PATH = os.path.join(
        BASE_DIR,
        "Symbolic-regression",
        "dat",
        "ReS2_rotate.dat"
    )

    OUTPUT_DATA_ROOT = os.path.join(
        BASE_DIR,
        "Symbolic-regression",
        "data_rotate"
    )

    DATABASE_PATH = os.path.join(
        BASE_DIR,
        "Symbolic-regression",
        "ReS2_rotate"
    )

    INCOSTEM_PATH = r"G:\Moire_Code\condition_ddpm\generate_data\generate_train_data\incostem.exe"

    # -------------------------------------------------------------------------
    # Generation settings.
    # -------------------------------------------------------------------------
    IMAGE_SIZE = 2048
    NO_SHOW_S = True

    # Clear previous DAT records to avoid mixing old and new paths.
    os.makedirs(os.path.dirname(DAT_PATH), exist_ok=True)

    with open(DAT_PATH, "w") as f:
        f.write("")

    os.makedirs(OUTPUT_DATA_ROOT, exist_ok=True)
    os.makedirs(DATABASE_PATH, exist_ok=True)

    # Optional: clear old archived database folders.
    for subfolder in ["layer1", "layer2", "bilayer"]:
        subfolder_path = os.path.join(DATABASE_PATH, subfolder)

        if os.path.exists(subfolder_path):
            import shutil
            shutil.rmtree(subfolder_path)

    # -------------------------------------------------------------------------
    # Twist-angle generation.
    # -------------------------------------------------------------------------
    for fixed_rot in range(0, 120, 60):
        generate_data(
            structure_path=STRUCTURE_PATH,
            DAT_path=DAT_PATH,
            move_list=[0, 0, fixed_rot],
            incostem_path=INCOSTEM_PATH,
            image_size=IMAGE_SIZE,
            image_num=1,
            no_show_s=NO_SHOW_S,
            output_data_root=OUTPUT_DATA_ROOT,
            database_path=DATABASE_PATH
        )

    print("ReS2 twist-stacked bilayer data generation finished.")

ReS2 twist-stacked bilayer data generation finished.


# ReS2 trilayer

In [7]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from ase.io import read
import torchvision.transforms.functional as TF
from collections import Counter
import subprocess
import cv2
import random
import numpy as np
from PIL import Image
import shutil


# =============================================================================
# 1. Database conversion
# =============================================================================

def to_database(database_path="", dat_path="", layer_suffix=""):
    """
    Convert simulated tif images and their corresponding labels into cropped png files.

    Parameters
    ----------
    database_path : str
        Root database path. Images and labels are saved into:
            database_path / layer_suffix / data
            database_path / layer_suffix / label

    dat_path : str
        Text file containing simulation folder paths. Each folder should contain:
            image/
            ReS2_<layer_suffix>.png

    layer_suffix : str
        Layer identifier:
            layer1
            layer2
            layer3
            trilayer

    Current behavior
    ----------------
    - Simulated tif images are converted into png images.
    - Image and label are both cropped into 1024 x 1024 patches.
    - Different layers are saved into separate folders to avoid mixing data.
    """
    layer_database_path = os.path.join(database_path, layer_suffix)

    data_dir = os.path.join(layer_database_path, "data")
    label_dir = os.path.join(layer_database_path, "label")

    os.makedirs(data_dir, exist_ok=True)
    os.makedirs(label_dir, exist_ok=True)

    with open(dat_path, "r") as data_paths:
        data_path = data_paths.readlines()

        for path in data_path:
            layer_dir = path.strip()

            label_path = os.path.join(layer_dir, f"ReS2_{layer_suffix}.png")
            if not os.path.exists(label_path):
                print(f"[Warning] Label file not found: {label_path}")
                continue

            label_n = Image.open(label_path)

            img_dir = os.path.join(layer_dir, "image")
            if not os.path.exists(img_dir):
                print(f"[Warning] Image directory not found: {img_dir}")
                continue

            for img in os.listdir(img_dir):
                img_path = os.path.join(img_dir, img)

                if not os.path.isfile(img_path):
                    continue

                if not img.lower().endswith((".tif", ".tiff")):
                    continue

                image = Image.open(img_path)

                # Keep the original crop logic: 1024 x 1024 crop.
                image = TF.crop(
                    image,
                    image.size[0] // 2 - 800,
                    image.size[1] // 2 - 300,
                    1024,
                    1024
                )

                label = TF.crop(
                    label_n.copy(),
                    label_n.size[0] // 2 - 800,
                    label_n.size[1] // 2 - 300,
                    1024,
                    1024
                )

                save_img_name = f"{os.path.splitext(img)[0]}_type_{layer_suffix}.png"
                save_label_name = f"{os.path.splitext(img)[0]}_type_{layer_suffix}.png"

                image.save(os.path.join(data_dir, save_img_name), "PNG")
                label.save(os.path.join(label_dir, save_label_name), "PNG")


# =============================================================================
# 2. XYZ export
# =============================================================================

def write_xyz(structure, file_name="", no_show_s=True):
    """
    Write an ASE Atoms object into the custom XYZ-like format required by incoSTEM.

    This version directly reads atom symbols and Cartesian coordinates from ASE.
    It does not create a temporary temp.xyz file.

    Parameters
    ----------
    structure : ase.Atoms
        Atomic structure to export.

    file_name : str
        Output xyz file path.

    no_show_s : bool
        If True, S atoms are omitted from the incoSTEM input.

    Element mapping
    ---------------
    Re -> 75
    S  -> 16
    """
    output_lines = []

    symbols = structure.get_chemical_symbols()
    atom_counts = Counter(symbols)

    formula = ""
    for atom, count in atom_counts.items():
        formula += atom + str(count)

    output_lines.append(f"{formula}\t{len(structure)}")

    cell = structure.get_cell()
    output_lines.append(f"{cell[0][0]}\t{cell[1][1]}\t{cell[2][2]}")

    atomic_number_map = {
        "Re": "75",
        "S": "16",
    }

    for atom in structure:
        symbol = atom.symbol

        if symbol == "S" and no_show_s:
            continue

        atomic_number = atomic_number_map.get(symbol, symbol)
        x, y, z = atom.position

        output_lines.append(
            f"{atomic_number}\t{x:.8f}\t{y:.8f}\t{z:.8f}\t1\t0"
        )

    output_lines.append("-1")

    with open(file_name, "w") as f:
        f.write("\n".join(output_lines))


# =============================================================================
# 3. Label generation
# =============================================================================

def save_label(size, structure, file_name, p_size=10):
    """
    Save a single-layer Re atom label.

    Parameters
    ----------
    size : int
        Reference image size.

    structure : ase.Atoms
        Single-layer ReS2 structure.

    file_name : str
        Output path without '.png'.

    p_size : int
        Marker radius for Re atoms.
    """
    if structure.cell[1][1] > structure.cell[0][0]:
        img_shape = [
            int(structure.cell[1][1] / (structure.cell[0][0] / size)),
            size
        ]
    else:
        img_shape = [
            size,
            int(structure.cell[0][0] / (structure.cell[1][1] / size))
        ]

    img = np.zeros(img_shape, np.uint8)

    for atom in structure:
        if atom.symbol == "Re":
            if structure.cell[1][1] > structure.cell[0][0]:
                p = atom.position[:2] / (structure.cell[0][0] / size)
            else:
                p = atom.position[:2] / (structure.cell[1][1] / size)

            p = p.astype(int)
            cv2.circle(img, p, p_size, 255, -1)

    cv2.imwrite(f"{file_name}.png", img)


def save_train_label(size, layer1, layer2, layer3, file_name, p_size=15):
    """
    Save a three-channel trilayer ReS2 label.

    Channel design
    --------------
    Channel 0: Re atoms in layer1.
    Channel 1: Re atoms in layer2.
    Channel 2: Re atoms in layer3.
    """
    if layer1.cell[1][1] > layer1.cell[0][0]:
        img_shape = [
            int(layer1.cell[1][1] / (layer1.cell[0][0] / size)),
            size
        ]
    else:
        img_shape = [
            size,
            int(layer1.cell[0][0] / (layer1.cell[1][1] / size))
        ]

    img = np.zeros(img_shape, np.uint8)

    img1 = img.copy()
    img2 = img.copy()
    img3 = img.copy()

    for atom in layer1:
        if atom.symbol == "Re":
            p = _get_pixel_pos(atom, layer1, size)
            cv2.circle(img1, p, p_size, 255, -1)

    for atom in layer2:
        if atom.symbol == "Re":
            p = _get_pixel_pos(atom, layer2, size)
            cv2.circle(img2, p, p_size, 255, -1)

    for atom in layer3:
        if atom.symbol == "Re":
            p = _get_pixel_pos(atom, layer3, size)
            cv2.circle(img3, p, p_size, 255, -1)

    merged_array = np.concatenate(([img1], [img2], [img3]), axis=0)
    label = np.transpose(merged_array, (1, 2, 0))

    cv2.imwrite(f"{file_name}.png", label)


def _get_pixel_pos(atom, layer, size):
    """
    Convert atomic xy position into pixel coordinates.
    """
    if layer.cell[1][1] > layer.cell[0][0]:
        p = atom.position[:2] / (layer.cell[0][0] / size)
    else:
        p = atom.position[:2] / (layer.cell[1][1] / size)

    return p.astype(int) + 1


# =============================================================================
# 4. incoSTEM execution
# =============================================================================

def run_computem(incostem_path, param_path):
    """
    Run incoSTEM simulations for all parameter files in a folder.

    Parameters
    ----------
    incostem_path : str
        Path to incostem.exe.

    param_path : str
        Folder containing parameter files.
    """
    for param in os.listdir(param_path):
        param_full_path = os.path.join(param_path, param)

        if not os.path.isfile(param_full_path):
            continue

        process = subprocess.Popen(
            incostem_path,
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            universal_newlines=True
        )

        with open(param_full_path, "r") as f:
            for line in f.readlines():
                process.stdin.write(line)
                process.stdin.flush()

        process.communicate()


# =============================================================================
# 5. incoSTEM parameter generation
# =============================================================================

def generate_param(path="", image_size=4096, params=["", "", "", ""], d_num=1, s_num=1, layer_type=""):
    """
    Generate incoSTEM parameter files for layer1, layer2, layer3 or trilayer ReS2.

    Parameters
    ----------
    path : str
        Folder of the current structure.

    image_size : int
        Simulated image size.

    params : list[str]
        Metadata used in image filename:
            params[0] = move_x1
            params[1] = move_y1
            params[2] = move_x2
            params[3] = move_y2

    d_num : int
        Number of defocus values.

    s_num : int
        Number of source-size values.

    layer_type : str
        One of:
            layer1
            layer2
            layer3
            trilayer
    """
    param_dir = os.path.join(path, "param")
    img_dir = os.path.join(path, "image")

    os.makedirs(param_dir, exist_ok=True)
    os.makedirs(img_dir, exist_ok=True)

    pd = 10.0 / d_num
    ps = 0.1 / s_num

    C12a = C12b = C21a = C21b = C23a = C23b = 0

    for i in range(d_num):
        for j in range(s_num):
            Defocus = 35 + i * pd
            Source_size = 0.6 + j * ps
            idx = random.uniform(10000, 20000)

            param_file = os.path.join(param_dir, f"{i}{j}_{layer_type}.param")

            xyz_file = os.path.join(path, f"ReS2_type_{layer_type}.xyz")

            img_file = os.path.join(
                img_dir,
                (
                    f"{params[0][:4]}_"
                    f"{params[1][:4]}_"
                    f"{params[2][:4]}_"
                    f"{params[3][:4]}_"
                    f"{str(Defocus)[:4]}_"
                    f"{str(Source_size)[:4]}_"
                    f"{str(idx)[:4]}_"
                    f"type_{layer_type}.tif"
                )
            )

            with open(param_file, "w") as f:
                f.write(f"{xyz_file}\n")
                f.write("1 1 1\n")
                f.write(f"{img_file}\n")
                f.write(f"{image_size} {image_size}\n")
                f.write(f"300 0 0 {Defocus} 21.3\n")
                f.write("39 200\n")
                f.write(
                    f"C12a {C12a} C12b {C12b} "
                    f"C21a {C21a} C21b {C21b} "
                    f"C23a {C23a} C23b {C23b} END\n"
                )
                f.write(f"{Source_size}\n")
                f.write("0\n")
                f.write("n\n")
                f.write("-1\n")


# =============================================================================
# 6. Trilayer ReS2 generation
# =============================================================================

def generate_data(
    structure_path="",
    DAT_path="",
    move_list1=None,
    move_list2=None,
    incostem_path="",
    image_size=2048,
    no_show_s=True,
    output_data_root="",
    database_path=""
):
    """
    Generate trilayer ReS2 and its corresponding layer1/layer2/layer3 images.

    Parameters
    ----------
    structure_path : str
        Path to the monolayer ReS2 structure file.

    DAT_path : str
        Global DAT file used to record generated main folders.

    move_list1 : list[float]
        In-plane displacement of layer2 relative to layer1:
            [move_x1, move_y1]

    move_list2 : list[float]
        In-plane displacement control for layer3:
            [move_x2, move_y2]

    incostem_path : str
        Path to incostem.exe.

    image_size : int
        Simulated image size.

    no_show_s : bool
        If True, S atoms are omitted from incoSTEM input.

    output_data_root : str
        Root folder for generated raw simulation folders.

    database_path : str
        Root database path. Final data are archived into:
            database_path/layer1/
            database_path/layer2/
            database_path/layer3/
            database_path/trilayer/

    Generated structures
    --------------------
    - layer1: original monolayer
    - layer2: translated second layer
    - layer3: translated third layer
    - trilayer: layer1 + layer2 + layer3
    """
    if move_list1 is None:
        move_list1 = [0, 0]

    if move_list2 is None:
        move_list2 = [0, 0]

    if output_data_root == "":
        raise ValueError("output_data_root must be provided.")

    if database_path == "":
        raise ValueError("database_path must be provided.")

    atoms = read(structure_path)

    ReS2_layer1 = atoms.copy()
    ReS2_layer2 = atoms.copy()
    ReS2_layer3 = atoms.copy()

    # Set interlayer spacing.
    ReS2_layer2.positions[:, 2] += 7.0
    ReS2_layer3.positions[:, 2] += 14.0

    # Apply in-plane displacement to layer2.
    move_x1, move_y1 = move_list1

    ReS2_layer2.positions[:, 0] += move_x1
    ReS2_layer2.positions[:, 1] += move_y1

    # Apply in-plane displacement to layer3.
    # The original logic is preserved:
    # layer3 displacement = move_list2 + 0.5 * layer2 displacement.
    move_x2, move_y2 = move_list2

    ReS2_layer3.positions[:, 0] += move_x2 + 0.5 * move_x1
    ReS2_layer3.positions[:, 1] += move_y2 + 0.5 * move_y1

    # Merge into trilayer and then re-split to avoid reference confusion.
    ReS2_trilayer = ReS2_layer1 + ReS2_layer2 + ReS2_layer3

    n_atoms = len(atoms)
    ReS2_layer1 = ReS2_trilayer[:n_atoms]
    ReS2_layer2 = ReS2_trilayer[n_atoms: 2 * n_atoms]
    ReS2_layer3 = ReS2_trilayer[2 * n_atoms:]

    main_dir = os.path.join(
        output_data_root,
        (
            f"ReS2_x1{str(move_x1)[:6]}_"
            f"y1{str(move_y1)[:6]}_"
            f"x2{str(move_x2)[:6]}_"
            f"y2{str(move_y2)[:6]}"
        )
    )

    os.makedirs(main_dir, exist_ok=True)
    os.makedirs(os.path.dirname(DAT_path), exist_ok=True)

    layers_config = [
        ("layer1", ReS2_layer1, "layer1"),
        ("layer2", ReS2_layer2, "layer2"),
        ("layer3", ReS2_layer3, "layer3"),
        ("trilayer", ReS2_trilayer, "trilayer")
    ]

    for layer_subdir, layer_struct, layer_type in layers_config:
        layer_dir = os.path.join(main_dir, layer_subdir)
        os.makedirs(layer_dir, exist_ok=True)

        xyz_file = os.path.join(layer_dir, f"ReS2_type_{layer_type}.xyz")

        write_xyz(
            layer_struct,
            file_name=xyz_file,
            no_show_s=no_show_s
        )

        label_file = os.path.join(layer_dir, f"ReS2_{layer_type}")

        if layer_type in ["layer1", "layer2", "layer3"]:
            save_label(
                image_size,
                layer_struct,
                label_file,
                p_size=10
            )
        else:
            save_train_label(
                image_size,
                ReS2_layer1,
                ReS2_layer2,
                ReS2_layer3,
                label_file,
                p_size=15
            )

        generate_param(
            path=layer_dir,
            image_size=image_size,
            params=[str(move_x1), str(move_y1), str(move_x2), str(move_y2)],
            d_num=1,
            s_num=1,
            layer_type=layer_type
        )

        run_computem(
            incostem_path,
            os.path.join(layer_dir, "param")
        )

        layer_dat = os.path.join(layer_dir, f"ReS2_{layer_type}.dat")

        with open(layer_dat, "w") as f:
            f.write(layer_dir + "\n")

        to_database(
            database_path=database_path,
            dat_path=layer_dat,
            layer_suffix=layer_type
        )

    with open(DAT_path, "a") as f:
        f.write(main_dir + "\n")


# =============================================================================
# 7. Execution section
# =============================================================================

if __name__ == "__main__":
    # -------------------------------------------------------------------------
    # Only modify paths here.
    # -------------------------------------------------------------------------
    BASE_DIR = r"G:\DiffStack-code"

    STRUCTURE_PATH = os.path.join(
        BASE_DIR,
        "Data_gen",
        "structure",
        "ReS2",
        "one_layer.xyz"
    )

    DAT_PATH = os.path.join(
        BASE_DIR,
        "Symbolic-regression",
        "dat",
        "ReS2_trilayer.dat"
    )

    OUTPUT_DATA_ROOT = os.path.join(
        BASE_DIR,
        "Symbolic-regression",
        "data_trilayer"
    )

    DATABASE_PATH = os.path.join(
        BASE_DIR,
        "Symbolic-regression",
        "ReS2_trilayer"
    )

    INCOSTEM_PATH = r"G:\Moire_Code\condition_ddpm\generate_data\generate_train_data\incostem.exe"

    # -------------------------------------------------------------------------
    # Generation settings.
    # -------------------------------------------------------------------------
    IMAGE_SIZE = 2048
    NO_SHOW_S = True

    # Fixed layer2 displacement.
    x1 = 1.6
    y1 = 1.5

    # Layer3 displacement control.
    x2 = 0 + 0.5 * x1
    y2 = 1.6 + 0.5 * y1

    # Clear previous DAT records to avoid mixing old and new paths.
    os.makedirs(os.path.dirname(DAT_PATH), exist_ok=True)

    with open(DAT_PATH, "w") as f:
        f.write("")

    os.makedirs(OUTPUT_DATA_ROOT, exist_ok=True)
    os.makedirs(DATABASE_PATH, exist_ok=True)

    # Optional: clear old archived database folders.
    for subfolder in ["layer1", "layer2", "layer3", "trilayer"]:
        subfolder_path = os.path.join(DATABASE_PATH, subfolder)

        if os.path.exists(subfolder_path):
            shutil.rmtree(subfolder_path)

    generate_data(
        structure_path=STRUCTURE_PATH,
        DAT_path=DAT_PATH,
        move_list1=[x1, y1],
        move_list2=[x2, y2],
        incostem_path=INCOSTEM_PATH,
        image_size=IMAGE_SIZE,
        no_show_s=NO_SHOW_S,
        output_data_root=OUTPUT_DATA_ROOT,
        database_path=DATABASE_PATH
    )

    print("ReS2 trilayer data generation finished.")

ReS2 trilayer data generation finished.
